# Reducing number of events for Mg22

Because of the way the Mg22 data is skewed, just "randomly" selecting a certain number of events from the total training set (to train and test with lower sample sizes) is not as random as you would expect.

This notebook first reduces the number of beam events (0 tracks) because it is a lot more common than the others and then it reduces the number of events in the 0,2, and 3 track data. The 2 and 4-5 track data is not as effected because there are so many fewer events for those tracks but this can be adjusted to affect those as well.

In [1]:
import numpy as np

In [2]:
training_set = np.load('Mg22_pretrain/voxel_data/Mg22_size512_convertXYZQ_train.npy')

In [3]:
training_set.shape
rand_shuffle = np.random.choice(len(training_set), len(training_set), replace = False)
split= 0
print(split)
train_data= training_set[rand_shuffle[split:],:,:]
print(train_data.shape)

0
(6000, 512, 7)


### Removing events from beam events
This is assuming we have a 60-20-20 split for 10000 events between train, val and test data repectively.
This removal of 1500 data is specific to this split but it might have to change depending on what the train data consists of.
Make sure to always check the shape of your training data to make sure you cut what you want to cut

In [16]:
# Assuming train_data is already defined and has shape (6000, 512, 7)

# Step 1: Create a boolean mask for the condition
mask = train_data[:, 0, 6] == 0 # change this number if a different number of tracks has an overwhelming number of events

# Step 2: Get the indices where the condition is true
indices_true = np.where(mask)[0]

# Step 3: Randomly sample 1500 indices from the true condition
sampled_indices = np.random.choice(indices_true, size=1500, replace=False) # you can replace size with whatever is necessary

# Step 4: Get the indices where the condition is false
indices_false = np.where(~mask)[0]

# Step 5: Combine the sampled indices with the false condition indices
final_indices = np.concatenate([sampled_indices, indices_false])

# Step 6: Create the final array by indexing train_data with final_indices
final_array = train_data[final_indices]

# Print the shape of the final array
print(f"Shape of the final array: {final_array.shape}")

# Verify the number of events with 0 as the first element of the 7th column
num_zero_events = np.sum(final_array[:, 0, 6] == 0)
print(f"Number of events with 0 as the first element of the 7th column: {num_zero_events}")

Shape of the final array: (4533, 512, 7)
Number of events with 0 as the first element of the 7th column: 1500


### Scaling down events for specific number of tracks

This will downsample the events for the number of tracks that you decide need to be downsampled. Change the `target_counts` as needed

In [110]:
# Assuming train_data is already defined and has shape (4500, 512, 7)

def selective_downsample(data, target_counts):
    # Get the unique classes and their counts
    classes, counts = np.unique(data[:, 0, 6], return_counts=True)
    
    # Initialize list to store indices for the final array
    final_indices = []
    
    for cls, count in zip(classes, counts):
        # Get indices for this class
        class_indices = np.where(data[:, 0, 6] == cls)[0]
        
        # If this class should be downsampled
        if cls in target_counts:
            # Randomly sample the target number of indices
            sampled_indices = np.random.choice(class_indices, size=target_counts[cls], replace=False)
            final_indices.extend(sampled_indices)
        else:
            # Keep all indices for this class
            final_indices.extend(class_indices)
    
    # Convert to numpy array and shuffle
    final_indices = np.array(final_indices)
    np.random.shuffle(final_indices)
    
    return data[final_indices]

# Define target counts for classes 0, 2, and 3
# Adjust these numbers as needed
target_counts = {
    0: 700,  # Downsample class 0 events
    2: 700,  # Downsample class 2 events
    3: 700  # Downsample class 3 events
}

# Perform selective downsampling
downsampled_data = selective_downsample(train_data, target_counts)

# Print the shape of the downsampled array
print(f"Shape of the downsampled array: {downsampled_data.shape}")

# Verify the distribution of classes
unique, counts = np.unique(downsampled_data[:, 0, 6], return_counts=True)
print("Class distribution after downsampling:")
for cls, count in zip(unique, counts):
    print(f"Class {cls}: {count}")

Shape of the downsampled array: (2167, 512, 7)
Class distribution after downsampling:
Class 0.0: 700
Class 1.0: 52
Class 2.0: 700
Class 3.0: 700
Class 4.0: 11
Class 5.0: 4


### Checking your new downsampled data

Just as precaution to make sure you didn't cut things you didn't want to cut

In [111]:
downsampled_data.shape

(2167, 512, 7)

In [112]:
train_f = downsampled_data[:,:,:6]
train_l = downsampled_data[:,0,6]
train_l = np.where(train_l == 5, 4, train_l)

In [113]:
num_events = train_l.shape[0]
print(num_events)
print(train_f.shape)
new_label, new_distr = np.unique(train_l, return_counts=True)
print(new_label)
print(new_distr)

2167
(2167, 512, 6)
[0. 1. 2. 3. 4.]
[700  52 700 700  15]


## Save your data

In [114]:
np.save(f'Mg22_dw_data/Mg22_size512_{num_events}train_features',train_f)
np.save(f'Mg22_dw_data/Mg22_size512_{num_events}train_labels',train_l)